In [1]:
import test_users
import numpy as np
import csv
from tqdm import tqdm
import os, pickle, json
import random
from surprise import SVDpp, Dataset, Reader
import pandas as pd
from scipy.sparse import csr_matrix
from collections import defaultdict


test_users = test_users.test_users

In [10]:
%pip install scikit-surprise pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


 ### Filmy i użytkownicy

 Poniższe klasy reprezentują koncepty filmu i użytkownika. Zauważ, że struktury danych ograniczają się do prostych identyfikatorów filmów i ocen wystawionych przez użytkowników, dodatkowo dla obydwóch typów tworzone są indeksy. Ta implementacja **nie** może ulec zmianie. Jeżeli chcesz mieć dostęp do innych danych (np. do gatunków filmów, albo do opisów niepochodzących ze źródła danych), umieść implementacje źródeł danych w swoim systemie rekomendacyjnym, a nie tutaj.

In [2]:
class Movie:
    index = {}
    name_index = {}
    inner_index = {}
    reverse_inner_index = {}
    inner_index_gen = 0
    def __init__(self, id, name):
        self.id = id
        self.name = name
        self.ratings = []
        self.genres = []
        Movie.index[id] = self
        Movie.name_index[name] = self
    def add_rating(self, rating):
        self.ratings.append(rating)
    
    
class User:
    index = {}
    def __init__(self, id):
        self.id = id
        self.ratings = {}
        User.index[id] = self
    def add_rating(self, movie, rating):
        movie.add_rating(rating)
        self.ratings[movie.id] = rating
    def __str__(self):
        str_bldr = f'{self.id}'
        return str_bldr


### Wczytanie danych

In [18]:
limit_ocen = 20000263 # maksymalnie 20000263

with open('data/movie.csv', encoding='utf-8') as file:
    csv_reader = csv.reader(file)
    next(csv_reader)
    for line in csv_reader:
        Movie(int(line[0]), line[1])

with open('data/rating.csv', encoding='utf-8') as file:
    csv_reader = csv.reader(file)
    next(csv_reader)
    
    for i, line in enumerate(tqdm(csv_reader, total=limit_ocen)):
        if i >= limit_ocen:
            break 
            
        user_id = int(line[0])
        movie_id = int(line[1])
        rating = float(line[2])
        
        if user_id not in User.index:
            User(user_id)
        User.index[user_id].add_rating(Movie.index[movie_id], rating)

100%|██████████| 20000263/20000263 [01:43<00:00, 192797.94it/s]


 ### Systemy oceniające - klasa bazowa

 Każdy system oceniający ma dostęp do wszystkich użytkowników i ocen, które nie są ocenami użytkowników testowych. Można również zaimplementować metody doboru innych danych (np. gatunków, tagów, danych zewnętrznych). Poniżej znajduje się klasa bazowa - w jej inicjalizatorze wczytywane są dotyczące ocen wystawionych przez użytkowników nie będących testowymi.

In [4]:
class RatingSystem:
    def __init__(self):
        self.users = {id : User.index[id] for id in User.index if id not in test_users}
        self.movie_ratings = {}
        for user in tqdm(self.users):
            for movie in self.users[user].ratings:
                if movie not in self.movie_ratings.keys():
                    self.movie_ratings[movie] = [self.users[user].ratings[movie]]
                else:
                    self.movie_ratings[movie].append(self.users[user].ratings[movie])
        
    def rate(self, user, movie):
        return

 ### Przykłady prostych systemów oceniających

In [15]:
class NaiveRating(RatingSystem):
    """
    Przykładowy system - naiwny. 
    Hipoteza: jeżeli zwrócę każdemu filmowi średnią ocenę (2.5/5), to moja ocena będzie niezła.
    """
    def __init__(self):
        super().__init__()
    def rate(self, user, movie):
        return 2.5
    def __str__(self):
        return 'Naive Rating'

class AverageMovieRating(RatingSystem):
    def __init__(self):
        super().__init__()
    def rate(self, user, movie):
        """
        Przykładowy system - średnia.
        Hipoteza: jeżeli zwrocę każdeu filmowi średnią ocenę (wynikającą z wszystkich ocen), to moja ocena będzie niezła.
        Jeżeli ten film jeszcze nie był oceniony, to zwrócę 2.5.
        """
        n = len(self.movie_ratings[movie])
        if n == 0:
            return 2.5
        else:
            return sum(self.movie_ratings[movie])/n
    def __str__(self):
        return 'Average Movie Rating'
class AverageUserRating(RatingSystem):
    def __init__(self):
        super().__init__()
    def rate(self, user, movie):
        """
        Przykładowy system - średnia użytkownika.
        Hipoteza: jeżeli zwrócę dla tego filmu średnią ocenę wystawioną przez użytkownika, to mój system będzie niezły.

        """
        n = len(user.ratings.values())
        if n == 0:
            return 2.5
        else:
            return sum(user.ratings.values())/n
    def __str__(self):
        return 'Average User Rating'

class GlobalAverageMovieRating(RatingSystem):
    def __init__(self):
        """
        Przykładowy system - średnia ocena filmu.
        Hipoteza: średnia ocena tego filmu wśród wszystkich użytkowników powinna być dobrą estymacją.
        """
        super().__init__()
        self.GlobalAverageMovieRating = 0
        self.TotalMovies = 0
        for movie in self.movie_ratings:
            for rating  in self.movie_ratings[movie]:
                self.GlobalAverageMovieRating += rating
                self.TotalMovies += 1
        self.GlobalAverageMovieRating /= self.TotalMovies

    def rate(self, user, movie):
        return self.GlobalAverageMovieRating
    def __str__(self):
        return 'Average Global Movie Rating'
    
class Cheater(RatingSystem):
    def __init__(self):
        super().__init__()

    def rate(self, user, movie):
        """
        Testowy system.
        Jeżeli ten system działa, to coś jest nie tak - systemy mają dostęp do ocen filmów, które mają wyznaczyć - powinien działać mniej więcej tak samo jak system naiwny.
        """
        if movie in user.ratings:
            return user.ratings[movie]
        else:
            return 2.5
    def __str__(self):
        return 'Cheater'



## SVD++ (Koren, 2008) 
Zwycięzca netflix prize
SVD++ rozszerza  SVD o tzw. implicit feedback -- uwzględnia sam fakt, że użytkownik ocenił dany film, nawet bez znajomości oceny. Dzięki temu model jest dokładniejszy niż klasyczny SVD, szczególnie dla użytkowników z wieloma ocenami.

In [ ]:
class SVD_156145_155941(RatingSystem):
    """
    Hiperparametry dobrane eksperymentalnie pod zbiór MovieLens 20M:
      - n_factors=50  : liczba cech ukrytych (kompromis jakość/czas)
      - n_epochs=25   : liczba iteracji SGD
      - lr_all=0.007  : learning rate (niższy niż domyślny, stabilniejszy)
      - reg_all=0.02  : regularyzacja L2 (zapobiega overfittingowi)

    Cold start fallback (gdy brak danych o userze/filmie):
      1. Średnia ocen danego filmu
      2. Średnia ocen danego użytkownika
      3. Globalna średnia wszystkich ocen
    """

    def __init__(self, n_factors: int = 50, n_epochs: int = 20,
                 lr_all: float = 0.007, reg_all: float = 0.02):
        super().__init__()

        self.n_factors = n_factors
        self.n_epochs = n_epochs
        self.lr_all = lr_all
        self.reg_all = reg_all

        self._movie_avg: dict[int, float] = {}
        self._user_avg: dict[int, float] = {}
        self._global_avg: float = 3.5

        self._prepare_and_train()
        
    def _prepare_and_train(self) -> None:
        print("[SVD++ by 155941 and 156145] Ekstrakcja ocen z datasetu...")

        from collections import defaultdict
        rows = []
        movie_sums: dict[int, list] = defaultdict(list)
        user_sums: dict[int, list] = defaultdict(list)
        total_sum = 0.0
        total_count = 0

        for u_id, user_obj in tqdm(self.users.items(), desc="Zgrywanie ocen", unit="user"):
            for m_id, rating in user_obj.ratings.items():
                rows.append((int(u_id), int(m_id), float(rating)))
                movie_sums[int(m_id)].append(float(rating))
                user_sums[int(u_id)].append(float(rating))
                total_sum += float(rating)
                total_count += 1

        if total_count > 0:
            self._global_avg = total_sum / total_count

        self._movie_avg = {m: sum(v) / len(v) for m, v in movie_sums.items()}
        self._user_avg = {u: sum(v) / len(v) for u, v in user_sums.items()}

        print(f"[SVD++ by 155941 and 156145] Załadowano {len(rows):,} ocen | globalna średnia: {self._global_avg:.4f}")

        df = pd.DataFrame(rows, columns=["userID", "movieID", "rating"])
        reader = Reader(rating_scale=(0.5, 5.0))
        data = Dataset.load_from_df(df[["userID", "movieID", "rating"]], reader)
        trainset = data.build_full_trainset()

        print(f"[SVD++ by 155941 and 156145] Trening(n_factors={self.n_factors}, "
              f"n_epochs={self.n_epochs}, lr={self.lr_all}, reg={self.reg_all})...")

        self.model = SVDpp(
            n_factors=self.n_factors,
            n_epochs=self.n_epochs,
            lr_all=self.lr_all,
            reg_all=self.reg_all,
            verbose=True,
        )
        self.model.fit(trainset)
        print("[SVD++ by 155941 and 156145] Model gotowy!")

    def rate(self, user, movie) -> float:
        """
        Zwraca przewidywaną ocenę dla pary (user, movie_id).

        Parametry zgodne z interfejsem RatingSystem:
          user  - obiekt User (posiada atrybut .id)
          movie - identyfikator filmu (int)

        Fallback kolejność gdy model nie ma danych:
          film średnia → user średnia → globalna średnia
        """
        try:
            user_id = user.id if hasattr(user, "id") else int(user)
            movie_id = movie if isinstance(movie, int) else int(movie)

            pred = self.model.predict(user_id, movie_id)

            # Gdy model zwraca fallback (was_impossible), użyj własnego
            if pred.details.get("was_impossible", False):
                return self._fallback(user_id, movie_id)

            return float(pred.est)

        except Exception as e:
            print(f"[SVD++ by 155941 and 156145] Prediction error: {e}")
            return self._global_avg

    def _fallback(self, user_id: int, movie_id: int) -> float:
        if movie_id in self._movie_avg:
            return self._movie_avg[movie_id]
        if user_id in self._user_avg:
            return self._user_avg[user_id]
        return self._global_avg

    def __str__(self) -> str:
        return "SVD++ by 155941 and 156145"  



## ALS

In [ ]:
class ALS_156145_155941(RatingSystem):
    """
    Hiperparametry:
      n_factors  = 40   -- liczba cech ukrytych (mniej niż SVD++ = szybciej)
      n_epochs   = 15   -- iteracji ALS (zbieżność szybsza niż SGD)
      reg        = 0.05 -- regularyzacja L2
    """

    def __init__(self, n_factors: int = 40, n_epochs: int = 15, reg: float = 0.05):
        super().__init__()
        self.n_factors = n_factors
        self.n_epochs = n_epochs
        self.reg = reg

        self._user_ids = []
        self._movie_ids = []
        self._u2i = {}
        self._m2i = {}

        self.global_avg = 3.5
        self.user_bias = None
        self.movie_bias = None
        self.U = None
        self.V = None

        self._movie_avg = {}
        self._user_avg = {}

        self._build()


    def _build(self):
        print("[Sys155941] Budowanie indeksów i macierzy ocen...")

        raw = []
        movie_sum = defaultdict(list)
        user_sum = defaultdict(list)

        for u_id, user_obj in tqdm(self.users.items(), desc="Ekstrakcja ocen", unit="user"):
            for m_id, rating in user_obj.ratings.items():
                raw.append((int(u_id), int(m_id), float(rating)))
                movie_sum[int(m_id)].append(float(rating))
                user_sum[int(u_id)].append(float(rating))

        all_ratings = [r for _, _, r in raw]
        self.global_avg = float(np.mean(all_ratings)) if all_ratings else 3.5
        self._movie_avg = {m: float(np.mean(v)) for m, v in movie_sum.items()}
        self._user_avg = {u: float(np.mean(v)) for u, v in user_sum.items()}

        self._user_ids = sorted(self.users.keys())
        self._movie_ids = sorted(movie_sum.keys())
        self._u2i = {uid: i for i, uid in enumerate(self._user_ids)}
        self._m2i = {mid: i for i, mid in enumerate(self._movie_ids)}

        n_users = len(self._user_ids)
        n_movies = len(self._movie_ids)

        print(f"[Sys155941] {len(raw):,} ocen | {n_users:,} userow | {n_movies:,} filmow")

        rows = np.array([self._u2i[u] for u, _, _ in raw], dtype=np.int32)
        cols = np.array([self._m2i[m] for _, m, _ in raw], dtype=np.int32)
        vals = np.array([r for _, _, r in raw], dtype=np.float32)
        R = csr_matrix((vals, (rows, cols)), shape=(n_users, n_movies))
        R_csc = R.tocsc()

        print("[Sys155941] Wyznaczanie biasow iteracyjnie...")
        self.user_bias = np.zeros(n_users, dtype=np.float32)
        self.movie_bias = np.zeros(n_movies, dtype=np.float32)

        for _ in tqdm(range(8), desc="Iteracje biasow"):
            for j in range(n_movies):
                col = R_csc.getcol(j)
                idxs = col.nonzero()[0]
                if len(idxs) == 0:
                    continue
                residuals = col.data - self.global_avg - self.user_bias[idxs]
                self.movie_bias[j] = residuals.sum() / (len(idxs) + self.reg * 10)

            for i in range(n_users):
                row = R.getrow(i)
                idxs = row.nonzero()[1]
                if len(idxs) == 0:
                    continue
                residuals = row.data - self.global_avg - self.movie_bias[idxs]
                self.user_bias[i] = residuals.sum() / (len(idxs) + self.reg * 10)

        print("[Sys155941] Trening ALS na czynnikach ukrytych...")

        res_vals = vals - self.global_avg - self.user_bias[rows] - self.movie_bias[cols]
        Res = csr_matrix((res_vals, (rows, cols)), shape=(n_users, n_movies))
        Res_csc = Res.tocsc()

        rng = np.random.default_rng(42)
        self.U = rng.standard_normal((n_users, self.n_factors)).astype(np.float32) * 0.01
        self.V = rng.standard_normal((n_movies, self.n_factors)).astype(np.float32) * 0.01
        reg_I = self.reg * np.eye(self.n_factors, dtype=np.float32)

        for _ in tqdm(range(self.n_epochs), desc="Epoki ALS"):
            for i in range(n_users):
                idxs = R.indices[R.indptr[i]:R.indptr[i+1]]
                if len(idxs) == 0: continue
                r = R.data[R.indptr[i]:R.indptr[i+1]]
                residuals = r - self.global_avg - self.movie_bias[idxs]
                self.user_bias[i] = residuals.sum() / (len(idxs) + self.reg * 10)

            for j in range(n_movies):
                idxs = R_csc.indices[R_csc.indptr[j]:R_csc.indptr[j+1]]
                if len(idxs) == 0: continue
                r = R_csc.data[R_csc.indptr[j]:R_csc.indptr[j+1]]
                residuals = r - self.global_avg - self.user_bias[idxs]
                self.movie_bias[j] = residuals.sum() / (len(idxs) + self.reg * 10)


        print("[Sys155941] Model gotowy!")

    # ------------------------------------------------------------------
    # Predykcja
    # ------------------------------------------------------------------

    def rate(self, user, movie):
        uid = user.id if hasattr(user, "id") else int(user)
        mid = movie if isinstance(movie, int) else int(movie)

        u_idx = self._u2i.get(uid)
        m_idx = self._m2i.get(mid)

        if u_idx is None or m_idx is None:
            return self._fallback(uid, mid)

        score = (
            self.global_avg
            + float(self.user_bias[u_idx])
            + float(self.movie_bias[m_idx])
            + float(self.U[u_idx] @ self.V[m_idx])
        )
        return float(np.clip(score, 0.5, 5.0))

    def _fallback(self, uid, mid):
        if mid in self._movie_avg:
            return self._movie_avg[mid]
        if uid in self._user_avg:
            return self._user_avg[uid]
        return self.global_avg

    def __str__(self):
        return "ALS System by 155941 and 156145"

 ### Ocena systemów

 Poniższe klasy dotyczą oceny systemów - zwróć uwagę na parametr verbose, służy on do ograniczania informacji zwrotnej

In [7]:
import copy
class RatingSystemCompetition:
    
    def __init__(self):
        self.registered_systems = []
        self.users = {id : User.index[id] for id in User.index if id not in test_users}
        self.verbose = 2
    def register(self, system):
        self.registered_systems.append(system)
        
    def build_round_robin(self):
        self.pairs = {}
        for system in self.registered_systems:
            self.pairs[system]  = []
            for competitor in self.registered_systems:
                if str(system) != str(competitor):
                    self.pairs[system].append((system, competitor))

            
    def runMatch(self, system, competitor):
        users_ids = np.random.choice(np.array(list(self.users.keys())), size=100)
        score = 0
        wins = 0
        loses = 0
        draws = 0
        for user_id in users_ids:
            user = self.users[user_id]
            user_copy = copy.deepcopy(self.users[user_id])
            movie_id = np.random.choice(np.array(list(user.ratings.keys())), size=1)[0]
            del user_copy.ratings[movie_id]
            true_rating = self.users[user_id].ratings[movie_id]
            system_rating = system.rate(user_copy,movie_id)
            competitor_rating = competitor.rate(user_copy,movie_id)
            
            if abs(true_rating - system_rating) <  abs(true_rating - competitor_rating):
                score += 1
                wins += 1
            elif abs(true_rating - system_rating) >  abs(true_rating - competitor_rating):
                score -= 1
                loses += 1
            else:
                draws += 1
                
        return score, wins, draws, loses
    
    def compete(self):
        self.total_scores = {}
        for system in self.pairs:
            self.total_scores[system] = 0
            if self.verbose >= 2: print(f'{system} analysis: ')
            for matchup in self.pairs[system]:
                score, wins, draws, loses = self.runMatch(matchup[0],matchup[1])
                if self.verbose >= 2: print(f'{matchup[0]} vs {matchup[1]} : {score} ({wins} wins, {draws} draws, {loses} loses)')
                self.total_scores[system] += score
            if self.verbose >= 2: print(f'{system} score: {self.total_scores[system]}')
        if self.verbose >= 1:
            print('Final scores: ')
            place = 1
            for system in sorted(self.total_scores, key=self.total_scores.get, reverse=True):
                print(f'{place}. {system}, {self.total_scores[system]} pkt')
                place += 1

 ### Przykładowa analiza

In [ ]:
competition = RatingSystemCompetition()

competition.register(SVD_156145_155941())       
competition.register(ALS_156145_155941())   

100%|██████████| 370/370 [00:00<00:00, 18394.54it/s]


[Sys155941] Ekstrakcja ocen z datasetu...


Zgrywanie ocen: 100%|██████████| 370/370 [00:00<00:00, 3081.92user/s]


[Sys155941] Załadowano 50,000 ocen | globalna średnia: 3.5063
[Sys155941] Trening SVD++ (n_factors=50, n_epochs=20, lr=0.007, reg=0.02)...
 processing epoch 0
 processing epoch 1
 processing epoch 2
 processing epoch 3
 processing epoch 4
 processing epoch 5
 processing epoch 6
 processing epoch 7
 processing epoch 8
 processing epoch 9
 processing epoch 10
 processing epoch 11
 processing epoch 12
 processing epoch 13
 processing epoch 14
 processing epoch 15
 processing epoch 16
 processing epoch 17
 processing epoch 18
 processing epoch 19
[Sys155941] Model gotowy!


100%|██████████| 370/370 [00:00<00:00, 20540.99it/s]


...

[Sys155941] 50,000 ocen | 370 userow | 6,471 filmow
[Sys155941] Wyznaczanie biasow iteracyjnie...


Iteracje biasow: 100%|██████████| 8/8 [00:04<00:00,  1.99it/s]


[Sys155941] Trening ALS na czynnikach ukrytych...


Epoki ALS: 100%|██████████| 15/15 [00:08<00:00,  1.80it/s]

[Sys155941] Model gotowy!
SVD++ System (155941) analysis: 
SVD++ System (155941) vs ALS+Bias System (155941) : -94 (3 wins, 0 draws, 97 loses)
SVD++ System (155941) score: -94
ALS+Bias System (155941) analysis: 
ALS+Bias System (155941) vs SVD++ System (155941) : 98 (99 wins, 0 draws, 1 loses)
ALS+Bias System (155941) score: 98
Final scores: 
1. ALS+Bias System (155941), 98 pkt
2. SVD++ System (155941), -94 pkt


In [ ]:
competition.build_round_robin()
competition.compete()

## Zapis modeli 

### Zapis SVD++

In [19]:
svd = SVD_156145_155941()
# svd = competition.registered_systems[0]

folder = f"saved_models/svdpp/{limit_ocen}"
os.makedirs(folder, exist_ok=True)

with open(f"{folder}/model.pkl", "wb") as f:
    pickle.dump(svd.model, f)

with open(f"{folder}/meta.json", "w") as f:
    json.dump({
        "global_avg": svd._global_avg,
        "movie_avg": {str(k): v for k, v in svd._movie_avg.items()},
        "user_avg":  {str(k): v for k, v in svd._user_avg.items()},
    }, f)

100%|██████████| 51738/51738 [00:02<00:00, 23891.45it/s]


[SVD++ by 155941 and 156145] Ekstrakcja ocen z datasetu...


Zgrywanie ocen: 100%|██████████| 51738/51738 [00:06<00:00, 8446.23user/s]


[SVD++ by 155941 and 156145] Załadowano 7,500,000 ocen | globalna średnia: 3.5249
[SVD++ by 155941 and 156145] Trening(n_factors=50, n_epochs=20, lr=0.007, reg=0.02)...
 processing epoch 0
 processing epoch 1
 processing epoch 2
 processing epoch 3
 processing epoch 4
 processing epoch 5
 processing epoch 6
 processing epoch 7
 processing epoch 8
 processing epoch 9
 processing epoch 10
 processing epoch 11
 processing epoch 12
 processing epoch 13
 processing epoch 14
 processing epoch 15
 processing epoch 16
 processing epoch 17
 processing epoch 18
 processing epoch 19
[SVD++ by 155941 and 156145] Model gotowy!


In [20]:
os.system("shutdown /s /f /t 60")

0

### Zapis ALS

In [19]:
als = ALS_156145_155941()
# als = competition.registered_systems[1]
folder = f"saved_models/als/{limit_ocen}"
os.makedirs(folder, exist_ok=True)

np.save(f"{folder}/U.npy", als.U)
np.save(f"{folder}/V.npy", als.V)
np.save(f"{folder}/user_bias.npy", als.user_bias)
np.save(f"{folder}/movie_bias.npy", als.movie_bias)

with open(f"{folder}/meta.json", "w") as f:
    json.dump({
        "global_avg": als.global_avg,
        "user_ids":   als._user_ids,
        "movie_ids":  als._movie_ids,
        "movie_avg":  {str(k): v for k, v in als._movie_avg.items()},
        "user_avg":   {str(k): v for k, v in als._user_avg.items()},
    }, f)


100%|██████████| 138493/138493 [00:09<00:00, 14657.03it/s]


[Sys155941] Budowanie indeksów i macierzy ocen...


Ekstrakcja ocen: 100%|██████████| 138493/138493 [00:36<00:00, 3777.45user/s]


[Sys155941] 20,000,263 ocen | 138,493 userow | 26,744 filmow
[Sys155941] Wyznaczanie biasow iteracyjnie...


Iteracje biasow: 100%|██████████| 8/8 [04:30<00:00, 33.81s/it]


[Sys155941] Trening ALS na czynnikach ukrytych...


Epoki ALS: 100%|██████████| 15/15 [00:49<00:00,  3.31s/it]


[Sys155941] Model gotowy!


### Wykorzystanie wytrenowanych modeli

#### Klasy dla zaczytania modeli 

In [11]:
class SVD_loaded(RatingSystem):
    def __init__(self, limit_ocen):
        super().__init__()
        folder = f"saved_models/svdpp/{limit_ocen}"
        
        with open(f"{folder}/model.pkl", "rb") as f:
            self.model = pickle.load(f)
        with open(f"{folder}/meta.json") as f:
            meta = json.load(f)
        
        self._global_avg    = meta["global_avg"]
        self._movie_avg     = {int(k): v for k, v in meta["movie_avg"].items()}
        self._user_avg      = {int(k): v for k, v in meta["user_avg"].items()}
        print("SVD++ wczytany!")

    def rate(self, user, movie) -> float:
        try:
            user_id  = user.id if hasattr(user, "id") else int(user)
            movie_id = movie if isinstance(movie, int) else int(movie)
            pred = self.model.predict(user_id, movie_id)
            if pred.details.get("was_impossible", False):
                return self._movie_avg.get(movie_id,
                       self._user_avg.get(user_id, self._global_avg))
            return float(pred.est)
        except:
            return self._global_avg

    def __str__(self):
        return "SVD++ loaded"


class ALS_loaded(RatingSystem):
    def __init__(self, limit_ocen):
        super().__init__()
        folder = f"saved_models/als/{limit_ocen}"

        self.U          = np.load(f"{folder}/U.npy")
        self.V          = np.load(f"{folder}/V.npy")
        self.user_bias  = np.load(f"{folder}/user_bias.npy")
        self.movie_bias = np.load(f"{folder}/movie_bias.npy")

        with open(f"{folder}/meta.json") as f:
            meta = json.load(f)

        self.global_avg  = meta["global_avg"]
        self._u2i        = {uid: i for i, uid in enumerate(meta["user_ids"])}
        self._m2i        = {mid: i for i, mid in enumerate(meta["movie_ids"])}
        self._movie_avg  = {int(k): v for k, v in meta["movie_avg"].items()}
        self._user_avg   = {int(k): v for k, v in meta["user_avg"].items()}
        print("ALS wczytany!")

    def rate(self, user, movie) -> float:
        uid   = user.id if hasattr(user, "id") else int(user)
        mid   = movie if isinstance(movie, int) else int(movie)
        u_idx = self._u2i.get(uid)
        m_idx = self._m2i.get(mid)
        if u_idx is None or m_idx is None:
            return self._movie_avg.get(mid,
                   self._user_avg.get(uid, self.global_avg))
        score = (self.global_avg + self.user_bias[u_idx]
                 + self.movie_bias[m_idx]
                 + float(self.U[u_idx] @ self.V[m_idx]))
        return float(np.clip(score, 0.5, 5.0))

    def __str__(self):
        return "ALS loaded"

### Wczytanie modeli

In [ ]:
limit_ocen = 20000263  # wartość musi być zgodna z tą, która była użyta do trenowania modeli (max 20M)

# SVD++
folder_svd = f"saved_models/svdpp/{limit_ocen}"

with open(f"{folder_svd}/model.pkl", "rb") as f:
    svdpp_model = pickle.load(f)

with open(f"{folder_svd}/meta.json") as f:
    svdpp_meta = json.load(f)

svdpp_global    = svdpp_meta["global_avg"]
svdpp_movie_avg = {int(k): v for k, v in svdpp_meta["movie_avg"].items()}
svdpp_user_avg  = {int(k): v for k, v in svdpp_meta["user_avg"].items()}

# ALS 
folder_als = f"saved_models/als/{limit_ocen}"

U          = np.load(f"{folder_als}/U.npy")
V          = np.load(f"{folder_als}/V.npy")
user_bias  = np.load(f"{folder_als}/user_bias.npy")
movie_bias = np.load(f"{folder_als}/movie_bias.npy")

with open(f"{folder_als}/meta.json") as f:
    als_meta = json.load(f)

als_global    = als_meta["global_avg"]
als_u2i       = {uid: i for i, uid in enumerate(als_meta["user_ids"])}
als_m2i       = {mid: i for i, mid in enumerate(als_meta["movie_ids"])}
als_movie_avg = {int(k): v for k, v in als_meta["movie_avg"].items()}
als_user_avg  = {int(k): v for k, v in als_meta["user_avg"].items()}

### Zarejestrowanie w konkursie

In [17]:
competition = RatingSystemCompetition()
competition.register(SVD_loaded(limit_ocen))
competition.register(ALS_loaded(limit_ocen))
competition.register(GlobalAverageMovieRating())
competition.register(NaiveRating())
competition.register(AverageMovieRating())
competition.register(AverageUserRating())
competition.register(Cheater())
competition.build_round_robin()
competition.compete()

100%|██████████| 51738/51738 [00:03<00:00, 13439.68it/s]


SVD++ wczytany!


100%|██████████| 51738/51738 [00:04<00:00, 11629.40it/s]


ALS wczytany!


100%|██████████| 51738/51738 [00:02<00:00, 23553.44it/s]


SVD++ loaded analysis: 
SVD++ loaded vs ALS loaded : 40 (70 wins, 0 draws, 30 loses)
SVD++ loaded vs Average Global Movie Rating : 46 (73 wins, 0 draws, 27 loses)
SVD++ loaded vs Naive Rating : 68 (84 wins, 0 draws, 16 loses)
SVD++ loaded vs Average Movie Rating : 38 (69 wins, 0 draws, 31 loses)
SVD++ loaded vs Average User Rating : 40 (70 wins, 0 draws, 30 loses)
SVD++ loaded vs Cheater : 60 (80 wins, 0 draws, 20 loses)
SVD++ loaded score: 292
ALS loaded analysis: 
ALS loaded vs SVD++ loaded : -31 (34 wins, 1 draws, 65 loses)
ALS loaded vs Average Global Movie Rating : 26 (63 wins, 0 draws, 37 loses)
ALS loaded vs Naive Rating : 60 (80 wins, 0 draws, 20 loses)
ALS loaded vs Average Movie Rating : 6 (53 wins, 0 draws, 47 loses)
ALS loaded vs Average User Rating : 26 (63 wins, 0 draws, 37 loses)
ALS loaded vs Cheater : 70 (85 wins, 0 draws, 15 loses)
ALS loaded score: 157
Average Global Movie Rating analysis: 
Average Global Movie Rating vs SVD++ loaded : -48 (26 wins, 0 draws, 74 loses